# WaterTAP tour — facet edition

A re-implementation of the classic `watertap-1` client walkthrough using the new
**facet-based** query API (`aq.graph()`), on the seawater-RO `model.ttl` in this folder.

Two symmetric moves plus introspection:

| move | meaning | rows |
|------|---------|------|
| `.facets()` | show what predicates/objects are reachable next | (read-only) |
| `.having(step, **φ)` | **stay** on the current nodes, keeping those with such an edge | never multiplies |
| `.follow(step, **φ)` | **move** the cursor to the neighbours along an edge | adds a column |

A **`Profile`** curates the discovery surface (hide noise, name virtual edges). Results come out
with `.count()` / `.nodes()` / `.frame()` / `.select(...)`, and — when the focus nodes are data
points — timeseries via `.data()` / `.dataframe()` / `.latest_data()`. Inspect any selection with
`.to_sparql()`.

## Connect

In [1]:
import polars as pl
from acquirium import Acquirium
from acquirium.Graframe import P, Profile, Reasoning, like

pl.Config.set_fmt_str_lengths(70)
pl.Config.set_tbl_rows(30)

acq = Acquirium(server_url="localhost", server_port=8000, use_ssl=False)
# g (the Graframe root) is built in the profile section below.

## Load the model

`insert_graph` reads the file client-side (path relative to this notebook).

In [2]:
acq.insert_graph("model.ttl", format="turtle", replace=True)
print("graph version:", acq.graph_version())

graph version: 34


## Curate the view with a profile

An ontology exposes far more predicates than any one task cares about. A `Profile` shapes the
*discovery surface* — which predicates/types show up in facets — and lets you **name virtual
edges** (property paths) so you traverse `follow("downstream")` instead of
`follow(P(connectedTo).plus())`.

`Profile.base()` hides schema noise (`rdf`/`rdfs`/`owl`/`sh`, class/shape objects); we layer the
water-domain predicates and a few named paths on top. Profiles shape discovery only — you can
still `follow`/`having` a hidden predicate explicitly, or pass `raw=True` to a facet call.

In [3]:
water = Profile.base().with_(
    # predicates worth seeing (namespace globs + a few exacts):
    allow=["s223:", "nawi:", "qudt:hasQuantityKind", "qudt:hasUnit", "s223:ofSubstance"],
    # ...minus the low-level connection plumbing:
    deny=[
        "s223:cnx", "s223:connected", "s223:connectedThrough", "s223:hasConnectionPoint",
        "s223:isConnectionPointOf", "s223:hasBoundaryConnectionPoint", "s223:connectsAt",
        "s223:connectsThrough", "s223:connectsTo", "s223:connectsFrom", "s223:connectedFrom",
    ],
    # named virtual edges (paths this domain actually cares about):
    edges={
        "downstream": "s223:connectedTo+",                      # transitive flow
        "upstream":   "^s223:connectedTo+",
        "measures":   "s223:hasProperty",                      # equipment to property
        "quantity":   "s223:hasProperty/qudt:hasQuantityKind", # equipment to quantity kind
    },
)

g = acq.graph(profile=water)

## Query by name (no ontology knowledge needed)

You don't have to know the URIs. **Every slot** — the class in `instances(...)`, the predicate in
`follow`/`having`, *and* the object in `value=` — resolves a natural-language string via
Acquirium's embedding matcher: `"pump"`→`nawi:Pump`, `"has property"`→`s223:hasProperty`,
`value="salt"`→`nawi:Constituent-Salt`. The rules are uniform:

- a full URI is used as-is;
- `prefix:local` with a **known** prefix expands as a CURIE;
- a **typo'd** prefix warns and falls back to fuzzy resolution of the local part;
- a colon-less string is treated as natural language.

`like(text, kind=...)` pins the concept kind when a bare word is ambiguous; a number or `Lit(...)`
forces a real literal. `suggest()` previews matches. (Pass `fuzzy=False` to `aq.graph()` to require
exact CURIEs.)

In [4]:
# Same queries as the rest of this tour, but by name:
print("pumps:", g.instances("pump").count())
# A linear multi-hop is one inline property path — segments resolve by name too:
print("pump quantity kinds:",
      g.instances("pump").follow("has property/has quantity kind").frame().to_series().to_list())

# Object values resolve by name too — no like() needed for an unambiguous word:
print("salt properties:", g.instances("observable property").having("of substance", value="salt").count())
# ...use like(text, kind=...) only to pin the kind when a bare word is ambiguous:
salt = g.instances("observable property").having("of substance", value=like("salt", "substance"))
print("salt properties (kind-pinned):", salt.count())

# Ambiguous term? preview and choose:
g.suggest("pump", kind="class")

2026-07-01 17:02:10,195 INFO acquirium.Graframe.resolve graframe: resolved 'pump' (class) -> http://data.ashrae.org/standard223#Pump
2026-07-01 17:02:13,235 INFO acquirium.Graframe.resolve graframe: resolved 'pump' (class) -> http://data.ashrae.org/standard223#Pump
2026-07-01 17:02:13,238 INFO acquirium.Graframe.resolve graframe: resolved 'has property' (predicate) -> http://data.ashrae.org/standard223#hasProperty
2026-07-01 17:02:13,241 INFO acquirium.Graframe.resolve graframe: resolved 'has quantity kind' (predicate) -> http://qudt.org/schema/qudt/hasQuantityKind


pumps: 3


2026-07-01 17:02:14,310 INFO acquirium.Graframe.resolve graframe: resolved 'observable property' (class) -> http://data.ashrae.org/standard223#ObservableProperty
2026-07-01 17:02:14,313 INFO acquirium.Graframe.resolve graframe: resolved 'of substance' (predicate) -> http://data.ashrae.org/standard223#ofSubstance
2026-07-01 17:02:14,480 INFO acquirium.Graframe.resolve graframe: resolved 'salt' -> urn:nawi-water-ontology#Salt-NaCl


pump quantity kinds: ['qk:Efficiency', 'qk:Power']


2026-07-01 17:02:15,496 INFO acquirium.Graframe.resolve graframe: resolved 'observable property' (class) -> http://data.ashrae.org/standard223#ObservableProperty
2026-07-01 17:02:15,499 INFO acquirium.Graframe.resolve graframe: resolved 'of substance' (predicate) -> http://data.ashrae.org/standard223#ofSubstance
2026-07-01 17:02:15,508 INFO acquirium.Graframe.resolve graframe: resolved 'salt' (substance) -> urn:nawi-water-ontology#Salt-NaCl


salt properties: 0
salt properties (kind-pinned): 0


[{'curie': 's223:Pump', 'score': 1.0, 'kind': 'class'},
 {'curie': 'nawi:Pump', 'score': 1.0, 'kind': 'class'},
 {'curie': 's223:HeatPump', 'score': 0.8464338779449463, 'kind': 'class'},
 {'curie': 's223:WaterToWaterHeatPump',
  'score': 0.7740627527236938,
  'kind': 'class'},
 {'curie': 's223:WaterToAirHeatPump',
  'score': 0.7534030675888062,
  'kind': 'class'}]

## Find entities by class

`g.instances(cls)` is the seed; it includes subclasses by default (the reasoning profile).

In [5]:
pumps = g.instances("nawi:Pump")
print("pumps:", pumps.count())
pumps.frame()

pumps: 3


focus
str
"""wbs:P1"""
"""wbs:P2"""
"""wbs:intake"""


In [6]:
# Raw firehose vs. the profiled view (named virtual edges surface at the top):
pumps.facets(raw=True).show(12)   # everything the ontology exposes
pumps.facets().show()             # curated + named edges (downstream/measures/...)

               Facets (by=predicate)                
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━┓
┃ dir ┃ predicate                ┃ support ┃ edges ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━┩
│ out │ s223:connectedTo         │       3 │     3 │
│ out │ rdfs:label               │       3 │     3 │
│ out │ s223:hasConnectionPoint  │       3 │     6 │
│ out │ s223:connectedThrough    │       3 │     5 │
│ out │ rdf:type                 │       3 │     3 │
│ out │ s223:cnx                 │       3 │     5 │
│ out │ rdfs:comment             │       3 │     3 │
│ out │ s223:connected           │       3 │     5 │
│ in  │ s223:hasMember           │       3 │     3 │
│ in  │ s223:isConnectionPointOf │       3 │     6 │
│ in  │ s223:connected           │       3 │     5 │
│ in  │ s223:cnx                 │       3 │     5 │
└─────┴──────────────────────────┴─────────┴───────┘

                    Facets (by=predicate)                    
┏━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━┓
┃ dir       ┃ predicate                   ┃ support ┃ edges ┃
┡━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━┩
│ ↳ virtual │ downstream                  │       3 │    37 │
│ ↳ virtual │ upstream                    │       2 │    23 │
│ ↳ virtual │ measures                    │       2 │     4 │
│ ↳ virtual │ quantity                    │       2 │     4 │
│ out       │ s223:connectedTo            │       3 │     3 │
│ in        │ s223:hasMember              │       3 │     3 │
│ out       │ s223:hasProperty            │       2 │     4 │
│ in        │ s223:connectedTo            │       2 │     2 │
│ in        │ s223:hasObservationLocation │       2 │     4 │
└───────────┴─────────────────────────────┴─────────┴───────┘

<Facets by=predicate rows=9>

## Follow relationships

`follow` walks an edge (moves the cursor). Because the profile named
`downstream = s223:connectedTo+`, you traverse it by name — no `P(...)` in sight. Filter the far
end inline with `is_a=` / `value=`.

In [7]:
# Everything reachable downstream of pump P1, and just the static mixers among them:
print("reachable downstream of P1:", g.nodes("wbs:P1").follow("downstream").count())
g.nodes("wbs:P1").follow("downstream", is_a="nawi:StaticMixer").frame()

reachable downstream of P1: 9


focus
str
"""wbs:co2-addition"""
"""wbs:lime-addition"""


## Property paths — multi-hop in one step

A `step` can be an inline **property path** in SPARQL syntax: `/` (sequence), `+` (transitive),
`^` (inverse), `|` (alternation). Each segment resolves by name too, so a *linear* multi-hop that
reaches a single filter needs no `P(...)` builder and no `where(lambda ...)` — the object filter
applies to the far end of the path. (Reach for `where` only for genuinely independent branches; a
named profile edge like `quantity` is the reusable equivalent of a path you use a lot.)

In [8]:
# Pumps that measure a pressure somewhere on their properties — ONE linear path,
# filtered at the far end. No P(...) builder, no where(lambda ...):
g.instances("nawi:Pump").having("s223:hasProperty/qudt:hasQuantityKind", value="pressure").frame()

# Path segments resolve by name too, and modifiers work inline:
print("pumps measuring pressure (by name):",
      g.instances("pump").having("has property/has quantity kind", value="pressure").count())
print("reachable downstream of P1 (transitive '+'):",
      g.nodes("wbs:P1").follow("s223:connectedTo+").count())

2026-07-01 17:02:26,970 INFO acquirium.Graframe.resolve graframe: resolved 'pressure' -> http://qudt.org/vocab/quantitykind/Pressure
2026-07-01 17:02:28,001 INFO acquirium.Graframe.resolve graframe: resolved 'pump' (class) -> http://data.ashrae.org/standard223#Pump
2026-07-01 17:02:28,004 INFO acquirium.Graframe.resolve graframe: resolved 'has property' (predicate) -> http://data.ashrae.org/standard223#hasProperty
2026-07-01 17:02:28,007 INFO acquirium.Graframe.resolve graframe: resolved 'has quantity kind' (predicate) -> http://qudt.org/schema/qudt/hasQuantityKind
2026-07-01 17:02:28,015 INFO acquirium.Graframe.resolve graframe: resolved 'pressure' -> http://qudt.org/vocab/quantitykind/Pressure


pumps measuring pressure (by name): 0
reachable downstream of P1 (transitive '+'): 9


## Data nodes (observable properties)

The measurable "data" are `s223:QuantifiableObservableProperty` nodes, attached to equipment via
`s223:hasProperty` (the named `measures` edge) and `observe`d by sensors.

In [9]:
props = g.instances("s223:QuantifiableObservableProperty")
print("observable properties:", props.count())
props.facets(by="predicate", direction="out").show()

observable properties: 29


             Facets (by=predicate)              
┏━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┳━━━━━━━┓
┃ dir ┃ predicate            ┃ support ┃ edges ┃
┡━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━╇━━━━━━━┩
│ out │ qudt:hasQuantityKind │      29 │    29 │
│ out │ qudt:hasUnit         │      19 │    19 │
│ out │ s223:ofMedium        │      13 │    13 │
│ out │ s223:ofSubstance     │       8 │     8 │
└─────┴──────────────────────┴─────────┴───────┘

<Facets by=predicate rows=4>

In [10]:
# The observable properties attached to a pump, via the named "measures" edge:
g.nodes("wbs:P1").follow("measures").frame()

focus
str
"""wbs:P1-efficiency"""
"""wbs:P1-mechanical-power"""


## Filter data nodes

`having` narrows the current set by an edge condition (stays put, never multiplies rows). The
classic `filter_by_quantity_kind` / `filter_by_unit` / `filter_by_substance` become `having(...)`
calls on the property's edges. Object values resolve by name, so you rarely need the exact URI —
and a facet row can be handed straight to `having` (see below).

In [11]:
# See what's actually available to filter on:
props.facets(by="pred-obj", direction="out", limit=40).to_polars().filter(
    pl.col("predicate").is_in(["qudt:hasQuantityKind", "qudt:hasUnit", "s223:ofSubstance"])
)

direction,predicate,object,support,edges
str,str,str,i64,i64
"""out""","""qudt:hasUnit""","""unit:KiloGM-PER-SEC""",10,10
"""out""","""qudt:hasQuantityKind""","""qk:MassFlowRate""",10,10
"""out""","""s223:ofSubstance""","""nawi:Constituent-Salt""",7,7
"""out""","""qudt:hasQuantityKind""","""qk:Pressure""",6,6
"""out""","""qudt:hasUnit""","""unit:UNITLESS""",3,3
"""out""","""qudt:hasQuantityKind""","""qk:Efficiency""",3,3
"""out""","""qudt:hasQuantityKind""","""qk:VolumeFlowRate""",2,2
"""out""","""qudt:hasQuantityKind""","""qk:Power""",2,2
"""out""","""qudt:hasUnit""","""unit:KiloGM-PER-M3""",2,2


In [12]:
# Facet rows are *actionable*: pick one and hand it straight to having()/follow() —
# the row already carries its predicate, direction, and object, so you never retype a URI.
f = props.facets(by="pred-obj", direction="out", limit=40)
pressure_row = f.row("qudt:hasQuantityKind", key="qk:Pressure")   # or f.row(<index>)
print("pressure points via facet row:", props.having(pressure_row).count())

pressure points via facet row: 6


In [13]:
# Values resolve by name — no exact CURIEs required:
print("Pressure           :", props.having("has quantity kind", value="pressure").count())
print("unit kg/s          :", props.having("qudt:hasUnit", value="unit:KiloGM-PER-SEC").count())

salt_flow = (props
    .having("of substance", value="salt")
    .having("qudt:hasUnit", value="unit:KiloGM-PER-SEC"))
print("salt mass flow (kg/s):", salt_flow.count())
salt_flow.frame()

2026-07-01 17:02:47,073 INFO acquirium.Graframe.resolve graframe: resolved 'has quantity kind' (predicate) -> http://qudt.org/schema/qudt/hasQuantityKind
2026-07-01 17:02:47,091 INFO acquirium.Graframe.resolve graframe: resolved 'pressure' -> http://qudt.org/vocab/quantitykind/Pressure


Pressure           : 6


2026-07-01 17:02:49,161 INFO acquirium.Graframe.resolve graframe: resolved 'of substance' (predicate) -> http://data.ashrae.org/standard223#ofSubstance
2026-07-01 17:02:49,178 INFO acquirium.Graframe.resolve graframe: resolved 'salt' -> urn:nawi-water-ontology#Salt-NaCl


unit kg/s          : 10
salt mass flow (kg/s): 0


focus
str


## Inspect the query

Every selection compiles to SPARQL — no black box.

In [14]:
print(salt_flow.to_sparql())

SELECT DISTINCT (?n0 AS ?focus)
WHERE {
  ?n0 <http://www.w3.org/1999/02/22-rdf-syntax-ns#type>/<http://www.w3.org/2000/01/rdf-schema#subClassOf>* <http://data.ashrae.org/standard223#QuantifiableObservableProperty> .
  FILTER EXISTS { ?n0 <http://data.ashrae.org/standard223#ofSubstance> ?n1 . VALUES ?n1 { <urn:nawi-water-ontology#Salt-NaCl> } }
  FILTER EXISTS { ?n0 <http://qudt.org/schema/qudt/hasUnit> ?n2 . VALUES ?n2 { <http://qudt.org/vocab/unit/KiloGM-PER-SEC> } }
}


## Systems

Systems are logical groupings of equipment/junctions/subsystems (`s223:hasMember`).

In [15]:
g.instances("s223:System").frame()

focus
str
"""wbs:pretreatment-system"""
"""wbs:desalination-system"""
"""wbs:posttreatment-system"""
"""wbs:seawater-ro-plant"""


In [16]:
# Hierarchy: systems that are members of other systems
(g.instances("s223:System").mark("system")
   .follow("s223:hasMember", is_a="s223:System").mark("subsystem")
   .select("system", "subsystem"))

system,subsystem
str,str
"""wbs:seawater-ro-plant""","""wbs:pretreatment-system"""
"""wbs:seawater-ro-plant""","""wbs:desalination-system"""
"""wbs:seawater-ro-plant""","""wbs:posttreatment-system"""


In [17]:
# Equipment count per system (direct members that are Equipment)
by_system = (g.instances("s223:System").mark("system")
              .follow("s223:hasMember", is_a="s223:Equipment").mark("equipment"))
(by_system.select("system", "equipment")
          .group_by("system")
          .agg(pl.col("equipment").count().alias("equipment_count"))
          .sort("equipment_count", descending=True))

system,equipment_count
str,u32
"""wbs:pretreatment-system""",9
"""wbs:posttreatment-system""",5
"""wbs:desalination-system""",4


In [18]:
# Pumps that are members of a specific system
g.nodes("wbs:pretreatment-system").follow("s223:hasMember", is_a="nawi:Pump").frame()

focus
str
"""wbs:intake"""


## All pumps and their observed properties

A join built with waypoints: mark the pump, hop out via `measures` to its properties (and their
quantity kind), mark those, then `select` the columns you want.

In [19]:
(g.instances("nawi:Pump").mark("pump")
   .follow("measures").mark("property")
   .follow("qudt:hasQuantityKind").mark("quantity")
   .select("pump", "property", "quantity"))

pump,property,quantity
str,str,str
"""wbs:P1""","""wbs:P1-efficiency""","""qk:Efficiency"""
"""wbs:P1""","""wbs:P1-mechanical-power""","""qk:Power"""
"""wbs:P2""","""wbs:P2-mechanical-power""","""qk:Power"""
"""wbs:P2""","""wbs:P2-efficiency""","""qk:Efficiency"""


## All data-generating entities within a system

System members → their properties → each property's quantity kind. (Desalination system, whose
equipment carry the observable properties directly.)

In [20]:
(g.nodes("wbs:desalination-system")
   .follow("s223:hasMember", is_a="s223:Equipment").mark("equipment")
   .follow("measures").mark("property")
   .follow("qudt:hasQuantityKind").mark("quantity")
   .select("equipment", "property", "quantity"))

equipment,property,quantity
str,str,str
"""wbs:P1""","""wbs:P1-efficiency""","""qk:Efficiency"""
"""wbs:P1""","""wbs:P1-mechanical-power""","""qk:Power"""
"""wbs:P2""","""wbs:P2-mechanical-power""","""qk:Power"""
"""wbs:P2""","""wbs:P2-efficiency""","""qk:Efficiency"""
"""wbs:RO""","""wbs:RO-membrane-area""","""qk:Area"""
"""wbs:PXR""","""wbs:PXR-efficiency""","""qk:Efficiency"""


## Pull timeseries

When the focus nodes are data points (they carry `ref:hasExternalReference`), `.data()` fetches
their timeseries as a `DataObject`; `.dataframe()` / `.latest_data()` are convenience wrappers.
Each mark becomes a context column — grouped with `.data().by("<mark>")` and present (under its
**bare name**) in the narrow `dataframe` / `metadata` frames — and series are auto-converted to
each point's unit.

> This seawater-RO `model.ttl` is metadata only (no external references), so the frame below is
> empty. Point at a deployment whose driver ingests data (e.g. the WATERTAP compose profile) and
> the *same code* returns one column per point.

In [21]:
# The pressure points, and their timeseries (wide: one column per point):
pressure = props.having("qudt:hasQuantityKind", value="qk:Pressure")
print("pressure points:", pressure.nodes())
pressure.dataframe(shape="wide")

pressure points: ['urn:swro/P1-out-pressure', 'urn:swro/PXR-brine-out-pressure', 'urn:swro/RO-in-pressure', 'urn:swro/RO-out-pressure', 'urn:swro/RO-out-retentate-pressure', 'urn:swro/conn-cartridge-filtration-to-S1-pressure']


data_alias,point_uri,ref_uri,time,value_numeric,value_text
str,str,str,"datetime[μs, UTC]",f64,str
